# Reasoning Prompts: Comparing Different Approaches

**Plain · Step by Step · Step Back · Vote**

I tested a loan-eligibility checker using four different prompting approaches. For each approach, I compared the accuracy, token usage, response time, and estimated cost.

I also tracked the results from each experiment to compare how different reasoning approaches affected the model's performance and cost.


### Step-by-Step Implementation :


## 1. Environment Setup

I set up the required environment and packages for this experiment.


In [1]:
import sys; sys.path[:0] = [".", "module01", "module02", "../module01", "../module02"]
import os
from pathlib import Path
try:
    from providers_utils import groq_client, ollama_client                                   # Session 11
    from prompt_utils import PromptSpec, init_work_repo, save_to_library                      # Session 14
    from prompt_tasks import TASK1, TASK2, TASK3, classify_spec                               # Session 14
    from reasoning_tasks import (DEMO, LOAN_CASES, LOAN_POLICY, OTHER, STRETCH, classify_cot_spec, problem_card,
                                 score_classifier_lab)
    from reasoning_utils import (DiskCache, Lab, Meter, accuracy_markdown, case_table, check_s15_work, comparison_table,
                                 pass_rate, pick_hardest, pick_local_chat_model, pick_trouble, run_strategy,
                                 save_accuracy_table, strategy_summary, verdict, with_scratchpad)
except ImportError as e:
    raise ImportError(f"{e}. This notebook uses files from Sessions 11, 14 and 15: run 'git pull' in the course folder, "
                      "then restart the kernel.") from e

MODEL = os.getenv("MODEL")
cache, meter = DiskCache(), Meter()
client = None
if os.getenv("BASE_URL") and os.getenv("API_KEY") and MODEL:
    client = groq_client()
lab = Lab(client, MODEL, cache=cache, meter=meter) if client else None
ready = lab is not None
print("model:", MODEL, "| ready:", ready)
print("answers are remembered in:", cache.path)
if not ready:
    print("No key in .env: the sections that call the model will say 'skipped'. Everything else still runs.")

results = {}                       # (problem id, strategy) -> Attempt, so re-running a cell replaces rather than duplicates
hardest = LOAN_CASES[4]            # replaced in section 5 by the case that gave the most trouble

def run(problem, strategy, **kw):
    a = run_strategy(lab, problem, strategy, **kw)
    results[(problem.id, a.strategy)] = a
    return a

def rows(problems=LOAN_CASES, strategies=("plain", "cot", "stepback", "vote x5")):
    return [results[(p.id, s)] for p in problems for s in strategies if (p.id, s) in results]

model: openai/gpt-oss-20b | ready: True
answers are remembered in: C:\Users\sailu\.genai_course\s15_cache.json


## 2. Token Usage Experiment

I started with a simple pricing calculation to examine how much output a reasoning model can use even when the requested answer is only one line.

The test uses a jacket priced at ₹2,000 with a 20% discount, followed by another 10% discount on the reduced price and a ₹100 coupon. The model was instructed to return only the final price without showing the calculation.

I measured the output tokens used for the response and compared the result with my expected token count.


In [2]:
print(problem_card(DEMO))
my_guess = 20          # YOUR TURN: how many OUTPUT tokens will a one-line answer cost? Change this, then run.
if ready:
    a = run(DEMO, "plain")
    print(f"answer: {a.raw}   | right answer: {DEMO.truth:g}   | correct: {a.correct}")
    print(f"output tokens: {a.completion_tokens:,}   (you guessed {my_guess})")
    if a.reasoning_tokens is not None:
        print(f"   thinking you never saw: {a.reasoning_tokens:,}      the visible reply: {a.completion_tokens - a.reasoning_tokens:,}")
    else:
        print("   (this model did not report any thinking tokens)")
    print(meter.report())
else:
    print("skipped (no key in .env)")

=== D0-jacket  (Retail): Two discounts and a coupon
A jacket is priced at Rs 2,000. The shop takes 20% off, then takes a further 10% off the REDUCED price, and then subtracts a Rs 100 coupon. What does the customer pay, in rupees?

answer: 1340   | right answer: 1340   | correct: True
output tokens: 116   (you guessed 20)
   thinking you never saw: 102      the visible reply: 14
spent so far: 0 calls (1 answered from memory), 0 tokens = about 0% of a 200,000-token free daily limit


### Token Usage Observation

The response was only one line, but the model still used more output tokens than expected. These tokens were used for the model's internal reasoning process, while the user only received the final answer.

This shows that asking a reasoning model to **“think step by step”** may add unnecessary reasoning and token usage when the model is already designed to reason internally.


## 3. Loan Eligibility Experiment

I tested a loan-eligibility checker on ten applicant cases. Each case was designed with a different condition, such as a boundary value, unit difference, or rule that could be easy to miss.

The correct result for each case was calculated independently using Python, so I could compare the model's answer with the expected result.

I tested the same cases using two prompting approaches:

* **Plain:** The model was asked to return only `FINAL: <answer>` without showing its reasoning.
* **Step by Step:** The model was asked to work through the calculations and then return `FINAL: <answer>`.

I compared both approaches based on how accurately they classified the ten cases and how much output they used.


In [3]:
print(LOAN_POLICY + "\n")
for c in LOAN_CASES:
    print(f"{c.id:<20}{c.text.splitlines()[1]}")
predict_wrong = []     # YOUR TURN: e.g. ["L05-age-at-end", "L10-months"]

Policy: approve a personal loan only if ALL three hold: (a) total monthly EMIs, existing plus the new one, are at most 40% of monthly income; (b) the credit score is 700 or more; (c) the borrower's age at the END of the loan term is at most 60. Otherwise decline.

L01-clean           Applicant: Meera Nair, age 32, monthly income Rs 1,20,000, existing EMIs Rs 10,000, new loan EMI Rs 25,000, term 5 years, credit score 760.
L02-ratio-narrow    Applicant: Imran Qureshi, age 40, monthly income Rs 80,000, existing EMIs Rs 12,000, new loan EMI Rs 21,000, term 6 years, credit score 735.
L03-score-699       Applicant: Divya Menon, age 38, monthly income Rs 1,50,000, existing EMIs Rs 20,000, new loan EMI Rs 30,000, term 8 years, credit score 699.
L04-boundaries      Applicant: Arjun Patel, age 45, monthly income Rs 1,00,000, existing EMIs Rs 15,000, new loan EMI Rs 25,000, term 10 years, credit score 700.
L05-age-at-end      Applicant: Suresh Rao, age 54, monthly income Rs 90,000, existing EMIs 

### Run Prompt Experiments

I ran both the **Plain Prompt** and **Step-by-Step Prompt** for each loan case and compared their accuracy, token usage, cost, and overall results.


In [21]:
if ready:
    for c in LOAN_CASES:
        run(c, "plain")
        run(c, "cot")
    print(case_table(results, LOAN_CASES, ("plain", "cot")))
    print()
    print(strategy_summary(rows(strategies=("plain", "cot")), MODEL))
    got_wrong = [a.problem_id for a in rows(strategies=("plain",)) if not a.correct]
    print("\nyou predicted plain would get wrong:", predict_wrong, "\nit actually got wrong:", got_wrong)
    print()
    print(meter.report())
else:
    print("skipped (no key in .env)")

case                right answer  plain             cot               
----------------------------------------------------------------------
L01-clean           approve       approve ok        approve ok        
L02-ratio-narrow    decline       decline ok        decline ok        
L03-score-699       decline       decline ok        decline ok        
L04-boundaries      approve       approve ok        approve ok        
L05-age-at-end      decline       decline ok        decline ok        
L06-age-60          approve       approve ok        approve ok        
L07-existing-emis   decline       decline ok        decline ok        
L08-two-fail        decline       decline ok        decline ok        
L09-large-numbers   approve       approve ok        approve ok        
L10-months          approve       approve ok        approve ok        

strategy     right  avg tokens  avg secs  avg $/1000 q
------------------------------------------------------
plain        10/10         476       

### Results Analysis

I compared the results of the **Plain** and **Step-by-Step** approaches using the prediction and experiment results.

I checked whether step-by-step reasoning improved the number of correct answers and how it affected token usage and cost. I then compared the change in accuracy with the change in cost to understand whether the additional reasoning was worth using for these cases.


## 4. Step-Back Prompting

I tested step-back prompting on the loan cases that were most difficult for the previous approaches.

In this approach, the model first identifies the general principles needed to solve the problem. These principles are then provided to the model in a second call, which uses them to solve the selected cases.

I compared this approach with the earlier prompting methods to see whether the additional step improved accuracy and how it affected token usage and cost.


In [22]:
trouble = pick_trouble(rows(strategies=("plain", "cot")), LOAN_CASES, 3)
print("step back on:", [c.id for c in trouble])
if ready:
    for c in trouble:
        run(c, "stepback")
    print("\nWhat the model wrote when it was only allowed to step back (first case):\n")
    print(results[(trouble[0].id, "stepback")].notes["principles"])
    print("\n" + case_table(results, trouble, ("plain", "cot", "stepback")))
    print()
    print(strategy_summary(rows(trouble, ("plain", "cot", "stepback")), MODEL))
    print()
    print(meter.report())
else:
    print("skipped (no key in .env)")

step back on: ['L03-score-699', 'L04-boundaries', 'L08-two-fail']

What the model wrote when it was only allowed to step back (first case):

- **EMI‑to‑income check**:  
  - Compute total monthly EMIs = existing EMI + new loan EMI.  
  - Compute 40 % of monthly income.  
  - Condition (a) holds if total EMIs ≤ 40 % of income.

- **Credit‑score threshold**:  
  - Condition (b) holds if credit score ≥ 700.

- **Age‑at‑term check**:  
  - Age at loan end = current age + loan term (in years).  
  - Condition (c) holds if this age ≤ 60.

- **All‑or‑nothing rule**:  
  - Approval requires *simultaneous* satisfaction of (a), (b), and (c).  
  - Failure of any single condition triggers a decline.

- **Common pitfalls**:  
  - Forgetting to add existing EMIs to the new EMI.  
  - Mis‑calculating 40 % of income (use decimal 0.40).  
  - Ignoring that the age check uses the *end* of the term, not the current age.

case                right answer  plain             cot               stepback     

### Step-Back Analysis

I reviewed the principles generated by the model and checked whether they added useful guidance or mainly repeated the loan policy already provided.

I also compared the step-back approach with step-by-step reasoning to understand where it could be more useful. Step-back prompting is more useful when a problem depends on identifying general rules or principles, while step-by-step prompting can be more suitable when the main challenge is working through detailed calculations or individual steps.


## 5. Self-Consistency

I tested self-consistency by asking the model the same step-by-step question multiple times with a temperature above zero. This allows the model to produce different answers for the same case.

I then selected the most common answer from the repeated responses. I applied this approach to the loan case that was most difficult in the previous experiments and compared the result with the other prompting approaches.

This experiment helps measure whether multiple reasoning attempts can improve reliability, and whether the improvement is worth the additional token usage and cost.


In [6]:
VOTES = 5
if ready:
    hardest = pick_hardest(rows(trouble, ("plain", "cot", "stepback")), trouble)
    print("voting on:", hardest.id, "-", hardest.title)
    a = run(hardest, "vote", n=VOTES)
    print("\nvotes:", a.notes["votes"], "| agreement:", f"{a.notes['agreement']:.0%}")
    print(f"single samples right: {a.notes['single_sample_correct']}/{VOTES}     the majority is right: {a.correct}")
    single = results.get((hardest.id, "cot"))
    if single:
        print(f"cost: {a.calls} calls, {a.tokens:,} tokens, {a.seconds:.0f}s   vs one step-by-step call: {single.tokens:,} tokens, {single.seconds:.0f}s")
    print()
    print(meter.report())
else:
    print("skipped (no key in .env)")

voting on: L04-boundaries - Exactly on two limits

votes: {'approve': 5} | agreement: 100%
single samples right: 5/5     the majority is right: True
cost: 5 calls, 3,705 tokens, 30s   vs one step-by-step call: 761 tokens, 1s

spent so far: 0 calls (32 answered from memory), 0 tokens = about 0% of a 200,000-token free daily limit


### Cost Analysis

I compared the estimated cost of each prompting strategy for 1,000 loan decisions and projected the cost for 200,000 decisions per month.

For the self-consistency approach, I also calculated the additional cost of making five calls for a single decision. I then considered how much improvement in accuracy would be needed to justify the extra cost.

This comparison helps identify situations where multiple reasoning attempts are useful and cases where the additional cost would not be justified.


## 6. Reasoning Effort

I tested the same plain loan-eligibility question using three reasoning effort levels: `low`, `medium`, and `high`.

The input and case remained the same; only the `reasoning_effort` setting was changed. I compared the results to see how the reasoning level affected the model's response, token usage, and cost.

If the selected model or provider does not support this setting, the experiment records that instead of producing an error.


In [7]:
dial = []
if ready:
    for effort in ("low", "high"):
        try:
            a = run_strategy(lab.variant(effort=effort), hardest, "plain")
            a.strategy = f"plain/{effort}"
            dial.append(a)
        except Exception as e:
            print(f"effort={effort!r} was not accepted by this model or provider: {str(e)[:120]}")
    if dial:
        print(comparison_table(dial, MODEL))
    print()
    print(meter.report())
else:
    print("skipped (no key in .env)")

problem               strategy      answer  result calls  tokens thinking   secs  $/1000 q
------------------------------------------------------------------------------------------
L04-boundaries        plain/low    approve  ok         1     316       53    1.6     $0.04
L04-boundaries        plain/high   approve  ok         1     574      311    7.1     $0.12

spent so far: 0 calls (34 answered from memory), 0 tokens = about 0% of a 200,000-token free daily limit


### Reasoning Effort Analysis

I compared the results from the `low`, `medium`, and `high` reasoning levels to see whether increasing the reasoning effort produced a better answer.

If different reasoning levels produced the same result, I checked whether the additional reasoning increased token usage or cost without improving the outcome. If the results were different, I considered which setting would be more suitable based on accuracy, consistency, and cost.

A single case is not enough to confirm that one setting is consistently better, so the result should be tested across multiple cases before choosing a setting for production use.


## 7. When Reasoning Does Not Help

I revisited the bank message classification experiment with eight messages and the same classification rules used earlier. The categories were `card`, `loan`, `account`, and `other`, with UPI problems classified as `account` and servicing requests classified as `other`.

I compared three approaches:

* **Plain:** Return only the predicted category.
* **Reasoning:** Use the same classification task with reasoning enabled.
* **Few-Shot:** Use four worked examples to show the model how the bank's classification rules should be applied.

I compared the results to see whether additional reasoning improved classification accuracy compared with a simpler prompt and with examples.


In [8]:
if ready:
    control = {
        "zero-shot, one word": score_classifier_lab(lab, classify_spec(0), mode="plain"),
        "zero-shot + thinking": score_classifier_lab(lab, classify_cot_spec(), mode="cot"),
        "four examples, one word": score_classifier_lab(lab, classify_spec(4), mode="plain"),
    }
    print(f"{'variant':<26}{'right':>7}{'tokens':>9}{'secs':>7}")
    for name, r in control.items():
        print(f"{name:<26}{r['accuracy']:>4}/{r['n']:<2}{r['tokens']:>9,}{r['seconds']:>7.1f}")
    print()
    for name, r in control.items():
        for text, truth, got in r["wrong"][:2]:
            print(f"  [{name}] wrong: {text[:52]!r} -> {got!r}, the bank says {truth!r}")
    print()
    print(meter.report())
else:
    print("skipped (no key in .env)")

variant                     right   tokens   secs
zero-shot, one word          6/8     1,539   10.4
zero-shot + thinking         6/8     1,879   12.6
four examples, one word      6/8     3,197   25.6

  [zero-shot, one word] wrong: 'I need to update my address after moving to a new fl' -> 'account', the bank says 'other'
  [zero-shot, one word] wrong: 'Please update the nominee details on my savings acco' -> 'account', the bank says 'other'
  [zero-shot + thinking] wrong: 'I need to update my address after moving to a new fl' -> 'account', the bank says 'other'
  [zero-shot + thinking] wrong: 'Please update the nominee details on my savings acco' -> 'account', the bank says 'other'
  [four examples, one word] wrong: 'I need to update my address after moving to a new fl' -> 'account', the bank says 'other'
  [four examples, one word] wrong: 'Please update the nominee details on my savings acco' -> 'account', the bank says 'other'

spent so far: 0 calls (58 answered from memory), 0 token

### Reasoning vs. Examples

I compared the classification results to see whether reasoning or worked examples had a greater effect on the incorrect labels.

The experiment showed the difference between **thinking about a rule** and **being given the rule**. Reasoning can help the model apply information it already has, but it cannot reliably recover a specific classification convention that was never provided.

For this type of task, worked examples are more useful when the main challenge is teaching the model a specific rule or convention. Reasoning is more useful when the required rules are already clear but the task requires more complex processing.


## 8. Local Llama Model

I tested a small local Llama model on five loan-eligibility cases using two prompting approaches:

* **Plain:** The model was asked to return only the final answer.
* **Step by Step:** The model was asked to work through the problem before giving the final answer.

The model runs locally through Ollama, so the experiment does not use API-based token billing. I compared the results to see whether asking the local model to reason step by step improved its accuracy.


In [9]:
LOCAL_IDS = ("L02", "L05", "L07", "L09", "L10")
local_cases = [c for c in LOAN_CASES if c.id.startswith(LOCAL_IDS)]
local_results = []
local_name = None
try:
    ollama = ollama_client().with_options(timeout=5, max_retries=0)
    local_name = pick_local_chat_model([m.id for m in ollama.models.list().data])
except Exception as e:
    print("Ollama is not reachable:", str(e)[:100])
if local_name:
    print("local model:", local_name)
    local = Lab(ollama_client(), local_name, max_tokens=1500, cache=cache, meter=Meter())
    for c in local_cases:
        for strategy in ("plain", "cot"):
            local_results.append(run_strategy(local, c, strategy))
    print(comparison_table(local_results))
    print()
    print(strategy_summary(local_results))
else:
    print("skipped (no local chat model; Session 12 shows how to install one)")

local model: llama3.2:3b
problem               strategy      answer  result calls  tokens thinking   secs  $/1000 q
------------------------------------------------------------------------------------------
L02-ratio-narrow      plain        decline  ok         1     208        -   16.3         -
L02-ratio-narrow      cot          decline  ok         1     416        -   26.7         -
L05-age-at-end        plain        decline  ok         1     207        -    3.7         -
L05-age-at-end        cot          decline  ok         1     461        -   31.7         -
L07-existing-emis     plain        approve  WRONG      1     206        -    3.8         -
L07-existing-emis     cot          decline  ok         1     481        -   34.3         -
L09-large-numbers     plain        decline  WRONG      1     209        -    3.9         -
L09-large-numbers     cot          decline  WRONG      1     477        -   33.6         -
L10-months            plain        decline  WRONG      1     207 

### Local vs. Hosted Model Analysis

I compared the local Llama model with the hosted model using the same loan cases and the same two prompting approaches.

I checked how much the accuracy changed when moving from the plain prompt to the step-by-step prompt for each model. This comparison helps show whether step-by-step prompting is consistently useful or whether its effect depends on the model.

The results also show why **“always think step by step”** is not a universal rule. The additional reasoning should be used when it provides a measurable improvement in accuracy or reliability that is worth the extra computation and cost.


## 9. Additional Domain Experiments

I extended the reasoning comparison to a few other problem types, including insurance claims, retail orders, and factory defect analysis.

I also included additional examples such as healthcare dosing with an invented formulary, telecom proration, and EMI calculations. These experiments help compare how different reasoning approaches perform across different types of tasks.

For the EMI calculation, the result can be calculated exactly with a tool, which provides a useful comparison for cases where a calculator is more reliable than model reasoning.


In [10]:
RUN_OTHER = False
if ready and RUN_OTHER:
    for p in OTHER + STRETCH:
        run(p, "plain")
        run(p, "cot")
    print(comparison_table(rows(OTHER + STRETCH, ("plain", "cot")), MODEL))
    print()
    print(meter.report())
else:
    print("skipped (set RUN_OTHER = True to run; needs a key)")

skipped (set RUN_OTHER = True to run; needs a key)


## 10. Reasoning Behind a Curtain

In a production system, the user should receive only the final answer, not the model's internal reasoning. I tested a structured response using `<scratchpad>` for internal working and `<final>` for the user-facing answer.

The code extracts only the `<final>` section before running the acceptance checks. The scratchpad is not displayed or stored as part of the customer-facing response.

I compared the original prompt with the structured version and ran both multiple times at a temperature above zero. This helped me evaluate whether separating internal reasoning from the final response improved the reliability and output format.


In [11]:
BASE = {TASK1.id: TASK1.reference, TASK2.id: TASK2.reference}      # YOUR TURN (optional): put your own v1 specs here
print("What version 2 adds to Task 2's standing rules:\n")
print(with_scratchpad(BASE[TASK2.id]).to_messages(TASK2.user_input)[0]["content"])

What version 2 adds to Task 2's standing rules:

You are a customer-communications writer at a retail bank.

Rules:
- Use at most 280 characters.
- Address the customer by first name.
- Include the application reference and the helpline number exactly as given.
- Mention the reapply period.
- Never reveal internal scores, cutoffs or the internal reason.
- Do your reasoning inside <scratchpad>...</scratchpad> first. The scratchpad is private working and is never shown to anyone.

Output format: A <scratchpad> block, then <final>...</final> containing exactly this: One plain-text SMS. No subject line, no markdown, no emojis.


In [12]:
RUNS = 2
compare = {}
if ready:
    for task in (TASK1, TASK2):
        v1 = pass_rate(lab, task, BASE[task.id], runs=RUNS)
        v2 = pass_rate(lab, task, BASE[task.id], hidden_reasoning=True, runs=RUNS)
        compare[task.id] = (v1, v2)
        print(f"{task.id}")
        print(f"   v1: {v1['avg_checks']:.1f} of {v1['of']} checks, every check passed in {v1['all_passed']}/{RUNS} runs, {v1['avg_tokens']:,.0f} tokens a call")
        print(f"   v2: {v2['avg_checks']:.1f} of {v2['of']} checks, every check passed in {v2['all_passed']}/{RUNS} runs, {v2['avg_tokens']:,.0f} tokens a call")
        print("   ->", verdict(v1, v2))
        if v2["any_leak"]:
            print("   !! the scratchpad MENTIONS the confidential values. That is why it is never shown to the customer.")
    print()
    print(meter.report())
else:
    print("skipped (no key in .env)")

T1-credit-summary
   v1: 4.0 of 4 checks, every check passed in 2/2 runs, 1,010 tokens a call
   v2: 4.0 of 4 checks, every check passed in 2/2 runs, 1,084 tokens a call
   -> v1 and v2 pass the same number of checks (4.0 of 4): no reason to switch, at 1.1x the tokens per call.
T2-sms-declined
   v1: 6.0 of 6 checks, every check passed in 2/2 runs, 820 tokens a call
   v2: 6.0 of 6 checks, every check passed in 2/2 runs, 986 tokens a call
   -> v1 and v2 pass the same number of checks (6.0 of 6): no reason to switch, at 1.2x the tokens per call.

spent so far: 0 calls (66 answered from memory), 0 tokens = about 0% of a 200,000-token free daily limit


### Scratchpad Security Analysis

I checked whether the internal scratchpad contained information that should not appear in the final response, such as the confidential score from Task 2.

I also considered access to the scratchpad in a production system, including which authorized internal users could access it, where it should be stored, and how long it should be retained.

Even if the structured prompt did not improve the acceptance-check results, keeping the separation between internal processing and the customer-facing answer can still be useful when the goal is to prevent sensitive information from being exposed.


## 11. Work Repository

I organized the results and improved prompts in my work repository.

Two artifacts were created:

1. **Accuracy Table** — `experiments/s15_accuracy_table.md`
   I recorded the results from the ten loan cases as a reference for comparing the different reasoning strategies.

2. **Prompt Library v2** — Updated prompts for Tasks 1 and 2
   I kept the original prompt versions unchanged and added new versions only when the experiment showed a measurable improvement. Each new version includes a `WHY` field describing the improvement using the experiment results, such as accuracy, token usage, and cost.

The updated prompts were saved only after the required experiments were completed and the results supported the change.


In [13]:
WORK_REPO = Path.home() / "Documents" / "genai-course-work"      # the same folder as Session 14
print("work repo:", WORK_REPO, "| exists:", WORK_REPO.exists())
if not WORK_REPO.exists():
    print("Clone your work repository first (module02/work_repo_steps.md), then rerun this cell.")

work repo: C:\Users\sailu\Documents\genai-course-work | exists: False
Clone your work repository first (module02/work_repo_steps.md), then rerun this cell.


In [14]:
SAVE_V2 = False
WHY = {TASK1.id: "",      # YOUR TURN: one line, with numbers from section 10
       TASK2.id: ""}
have_ten = all((c.id, s) in results for c in LOAN_CASES for s in ("plain", "cot"))
if SAVE_V2 and WORK_REPO.exists() and have_ten and compare and all(WHY.values()):
    for task in (TASK1, TASK2):
        run_v2 = compare[task.id][1]["rows"][0]
        save_to_library(WORK_REPO, task.id, 2, with_scratchpad(BASE[task.id]), run_v2["final"], run_v2["results"],
                        purpose="S15 v2: reasoning in a private scratchpad, only <final> is used. " + WHY[task.id])
    table = accuracy_markdown(results, LOAN_CASES, ("plain", "cot", "stepback", "vote x5"), MODEL)
    path = save_accuracy_table(WORK_REPO, table)
    print("saved 2 version-2 entries to", WORK_REPO / "prompts" / "library.md")
    print("saved the accuracy table to", path)
else:
    print("not saved (SAVE_V2 is False, or the work repo, sections 3 and 10 results, or a WHY line is missing)")

not saved (SAVE_V2 is False, or the work repo, sections 3 and 10 results, or a WHY line is missing)


In [15]:
for r in check_s15_work(WORK_REPO):
    print(("PASS  " if r.passed else "FAIL  ") + r.name)
    if not r.passed and r.detail:
        print("        fix:", r.detail)

FAIL  prompts/library.md exists in your work repo
        fix: run the Session 14 notebook cells first
FAIL  experiments/s15_accuracy_table.md exists
        fix: set SAVE_V2 = True in section 11 and run that cell once


## 12. Reasoning Strategy Decision Table

Based on the experiments, I summarized when each reasoning strategy is useful.

| Type of Problem                                      | Plain | Step by Step   | Step Back                 | Vote ×5                   |
| ---------------------------------------------------- | ----- | -------------- | ------------------------- | ------------------------- |
| Several rules chained, nothing hidden (L02, L07)     | use   | use            | skip                      | only if accuracy improves |
| A rule that fails quietly (L05, L10)                 | skip  | use            | only if rules are unclear | use                       |
| Values exactly on a limit (L03, L04, L06)            | use   | use            | skip                      | only if errors occur      |
| A formula or a piece of theory to recall (R4)        | use   | only if needed | use                       | skip                      |
| Classification with specific rules (bank complaints) | use   | skip           | skip                      | only if inconsistent      |
| A calculation a calculator can do exactly (S3, EMI)  | skip  | skip           | skip                      | skip                      |

The main takeaway is that reasoning should be used when it provides a clear benefit. For exact calculations, a calculator or tool is more reliable than repeated model reasoning. For classification tasks with specific rules, providing the rules or examples is often more useful than asking the model to reason more.

